# Quadratic Formula — Sparse Symbolic Regression via FISTA

This notebook recovers the quadratic formula (Section 5.5 of the paper) from data, using FISTA for optimization.
Because our hypothesis has a square root in it, we calculate the gradient by hand, using the chain rule through the square root.

In [11]:
import numpy as np


In [12]:
#     FISTA: soft-thresholding + Nesterov momentum.
#     Normally FISTA uses one fixed step size, computed from a matrix (the Lipschitz constant).
#     Our model is nonlinear, so there is no single matrix to compute this from.
#     Instead, we find a good step size at every iteration using backtracking.

def soft_threshold(z, thresh):
    return np.sign(z) * np.maximum(np.abs(z) - thresh, 0.0)

def fista(f_smooth, grad_fn, x0, lam, max_iter=15000, L0=10.0, eta=2.0, tol=1e-14):
    """Minimizes f_smooth(x) + lam*||x||_1. grad_fn(x) must return the exact gradient of
    f_smooth at x (supplied by hand below)."""
    x = np.array(x0, dtype=float)
    y = x.copy()
    t = 1.0
    L = L0
    for _ in range(max_iter):
        fy, gy = f_smooth(y), grad_fn(y)
        L = max(L / eta, L0 * 1e-6)
        while True:
            step = 1.0 / L
            x_new = soft_threshold(y - step * gy, lam * step)
            diff = x_new - y
            Q = fy + np.dot(gy, diff) + (L / 2.0) * np.dot(diff, diff)
            if f_smooth(x_new) <= Q + 1e-12 or L > 1e18:
                break
            L *= eta
        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        y = x_new + ((t - 1) / t_new) * (x_new - x)
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            break
        x, t = x_new, t_new
    return x


In [14]:
#    Data generation (Newton's method).

num_params = 10
num_samples = 100

def newton_root(b, c, x0, max_iter=50, tol=1e-10):
    x = x0
    for _ in range(max_iter):
        f, df = x**2 + b*x + c, 2*x + b
        if abs(df) < 1e-12:
            return None
        x_new = x - f / df
        if abs(x_new - x) < tol:
            return x_new
        x = x_new
    return None

def make_branch_data(x0_newton, seed):
    np.random.seed(seed)
    b_vals, c_vals, roots = [], [], []
    while len(b_vals) < num_samples:
        b, c = np.random.uniform(-10, 10), np.random.uniform(-10, 10)
        if b**2 - 4*c > 0:
            root = newton_root(b, c, x0=x0_newton)
            if root is not None:
                b_vals.append(b); c_vals.append(c); roots.append(root)
    return np.array(b_vals), np.array(c_vals), np.array(roots)

b_pos, c_pos, u_pos = make_branch_data(x0_newton=10, seed=1)    # + branch
b_neg, c_neg, u_neg = make_branch_data(x0_newton=-10, seed=1)   # - branch
print(f"Generated {len(u_pos)} samples per branch.")


Generated 100 samples per branch.


In [15]:
#    Hypothesis (eq. 10) and inner term S (eq. 11)

def hypothesis(xi, b, c):
    S = xi[4] + xi[5]*b + xi[6]*c + xi[7]*b**2 + xi[8]*c**2 + xi[9]*b*c
    S = np.maximum(S, 1e-8)   # keep the argument of sqrt non-negative during optimization
    return xi[0] + xi[1]*b + xi[2]*c + xi[3]*np.sqrt(S)

def make_objective(b, c, u):
    def objective(xi):
        return np.mean(0.5 * (hypothesis(xi, b, c) - u) ** 2)
    return objective


In [16]:
#     This is the gradient. We calculate it by hand, using the chain rule through sqrt(S).
#     Formula: d(loss)/d(xi_k) = mean of [residual * d(xhat)/d(xi_k)]
#     where residual = xhat - u (the prediction error)
#     xi0, xi1, xi2 are outside the sqrt -> their derivatives are simply 1, b, and c
#     xi3 multiplies sqrt(S)             -> its derivative is just sqrt(S)
#     xi4 to xi9 are inside S            -> we reach them through the sqrt, so their
#                                            derivative is xi3 / (2*sqrt(S)) times dS/d(xi_k)
#
#     Note: we clip S so it never goes negative (np.maximum). Where S was clipped,
#     the derivative is 0 -- that's what 'mask' does below.

def make_grad(b, c, u):
    def grad_fn(xi):
        S_raw = xi[4] + xi[5]*b + xi[6]*c + xi[7]*b**2 + xi[8]*c**2 + xi[9]*b*c
        mask = (S_raw > 1e-8).astype(float)
        sqrtS = np.sqrt(np.maximum(S_raw, 1e-8))
        residual = xi[0] + xi[1]*b + xi[2]*c + xi[3]*sqrtS - u
        dS_dxi = {4: np.ones_like(b), 5: b, 6: c, 7: b**2, 8: c**2, 9: b*c}
        g = np.zeros(10)
        g[0] = np.mean(residual)
        g[1] = np.mean(residual * b)
        g[2] = np.mean(residual * c)
        g[3] = np.mean(residual * sqrtS)
        for k in range(4, 10):
            g[k] = np.mean(residual * xi[3] * mask * dS_dxi[k] / (2 * sqrtS))
        return g
    return grad_fn

objective_pos, grad_pos = make_objective(b_pos, c_pos, u_pos), make_grad(b_pos, c_pos, u_pos)
objective_neg, grad_neg = make_objective(b_neg, c_neg, u_neg), make_grad(b_neg, c_neg, u_neg)


In [17]:
#    Fit both branches with FISTA (multiple random restarts, as the problem is non-convex) 

def fit_branch(objective, grad_fn, n_restarts=8, max_iter=15000, lam=1e-5, L0=10.0, seed=0):
    np.random.seed(seed)
    best_xi, best_err = None, np.inf
    for _ in range(n_restarts):
        x0 = np.random.randn(num_params)
        xi = fista(objective, grad_fn, x0, lam=lam, max_iter=max_iter, L0=L0)
        err = float(objective(xi))
        if err < best_err:
            best_err, best_xi = err, xi
    return best_xi, best_err


In [18]:
xi_pos, err_pos = fit_branch(objective_pos, grad_pos, seed=100)
print("+ branch: training MSE =", err_pos)


+ branch: training MSE = 2.7040382765334885e-10


In [19]:
xi_neg, err_neg = fit_branch(objective_neg, grad_neg, seed=200)
print("- branch: training MSE =", err_neg)


- branch: training MSE = 1.0908126596133905e-08


In [20]:
#    Scale normalization + hard thresholding 

def normalize(xi):
    xi = xi.copy()
    xi7 = xi[7]                          # coefficient of b^2 inside S, forced to +1
    scale = np.sqrt(abs(xi7))
    xi[3] *= scale
    xi[4:10] /= abs(xi7)
    xi[7] = np.sign(xi7)
    return xi

def threshold(xi, tol=1e-2):
    xi = xi.copy()
    xi[np.abs(xi) < tol] = 0.0
    return xi

xi_pos_final = threshold(normalize(xi_pos))
xi_neg_final = threshold(normalize(xi_neg))


In [21]:
#   Final result (matches Table 3 of the paper) 

print(f"{'Coefficient':<26}{'Recovered (+)':<16}{'Recovered (-)':<16}{'Exact':<8}")
print(f"{'xi1 (coeff. of b)':<26}{xi_pos_final[1]:<16.3f}{xi_neg_final[1]:<16.3f}{-0.5:<8}")
print(f"{'xi3 (coeff. of sqrt(S))':<26}{xi_pos_final[3]:<16.3f}{xi_neg_final[3]:<16.3f}{'+/-0.5':<8}")
print(f"{'xi6 (coeff. of c in S)':<26}{xi_pos_final[6]:<16.3f}{xi_neg_final[6]:<16.3f}{-4.0:<8}")
print(f"{'xi7 (coeff. of b^2 in S)':<26}{xi_pos_final[7]:<16.3f}{xi_neg_final[7]:<16.3f}{1.0:<8}")
print()
print("All remaining coefficients (+ branch):", xi_pos_final[[0,2,4,5,8,9]])
print("All remaining coefficients (- branch):", xi_neg_final[[0,2,4,5,8,9]])


Coefficient               Recovered (+)   Recovered (-)   Exact   
xi1 (coeff. of b)         -0.500          -0.500          -0.5    
xi3 (coeff. of sqrt(S))   0.500           -0.500          +/-0.5  
xi6 (coeff. of c in S)    -4.000          -3.999          -4.0    
xi7 (coeff. of b^2 in S)  1.000           1.000           1.0     

All remaining coefficients (+ branch): [0. 0. 0. 0. 0. 0.]
All remaining coefficients (- branch): [0. 0. 0. 0. 0. 0.]
